# Pipeline v14 — CoT LoRA Fine-tune trên `explanation` field
**EXACT 2026 — XAI Challenge @ IJCNN | Qwen3-8B + Unsloth QLoRA | Kaggle T4×2**

## Vì sao v14 (và vì sao đây là đòn duy nhất còn lại)

Bằng chứng từ v13.3 (oracle 63.5%, best aggregator 53%, pure_qwen-CoT base 33%):
- Z3 bucket đã đụng trần ngữ nghĩa ~60%, không vượt qua được bằng entailment cổ điển
- Aggregator bất kỳ trên signals hiện tại chỉ chạm 53-54%
- **Đòn bẩy duy nhất còn lại: nâng chất lượng Qwen-CoT signal** — base 33% là quá thấp

Dataset có **`explanation` field** — 822 reasoning chains gold-supervised tham chiếu trực tiếp premise number. Đây là *chính xác* pedagogical reasoning mà nhãn dataset thưởng (gồm cả những fallacious-but-labeled chain). **Chưa khai thác đến v14.**

## Vì sao v14 không lặp lại sai lầm v6

| | v6 (failed) | v14 |
|---|-------------|-----|
| Target | JSON AST output | Free-text reasoning + `Final Answer: X` |
| Hệ quả | Kill reasoning (alignment tax) | **Train ĐÚNG cái Qwen cần làm** |
| Format | Cấu trúc cứng | Tự nhiên (matches dataset style) |
| Mất khả năng gì | Đa bước CoT | Không — đây là tăng cường, không thay thế |

## Data prep
- 411 samples × 2 questions/sample = 822 cặp `(premises, question, explanation, gold)`
- Train/val split sample-level với **CAL_SEED=42, 80/20** — KHỚP HOÀN TOÀN với v13.3 → val accuracy của LoRA-Qwen có thể so trực tiếp với pure_qwen 33% trên cùng 163 câu
- Training target = `explanation + "\n\nFinal Answer: <gold>"` (append nếu explanation chưa có marker — chỉ 160/812 đã có)

## Hyperparams
- Unsloth QLoRA 4-bit | r=16, α=16, dropout=0
- LR 1e-4, warmup 10, weight_decay 0.01
- 2 epochs × ~328 examples/epoch ≈ 656 steps
- Batch 2 × grad_accum 4 = effective batch 8
- max_seq_length 4096 (mean ~700 tok, max ~1100)

## Mục tiêu định lượng
- Qwen-CoT val acc: **33% → ≥ 50%** (success threshold)
- Nếu đạt: cắm LoRA vào v13.3 pipeline → kỳ vọng tổng accuracy 53% → 60-65%

## Inference compatibility
- Train với `enable_thinking=False` (reasoning trực tiếp trong assistant text)
- Lưu LoRA adapter format PEFT → vLLM nạp được qua `enable_lora=True`
- Inline eval cuối notebook dùng Unsloth FastLanguageModel.for_inference để verify ngay

## Output
- `/kaggle/working/qwen3_cot_lora_v14/` (PEFT adapter — dùng cho v13.3 sau)
- `/kaggle/working/v14_eval_results.json` (val accuracy + per-question results)


In [ ]:
#!/usr/bin/env python3
"""
notebook_v14_cot_lora_finetune.py
EXACT 2026 -- v14: Train Qwen3-8B LoRA on dataset's `explanation` field
to produce pedagogical CoT reasoning + Final Answer marker.
Output: /kaggle/working/qwen3_cot_lora_v14 (PEFT adapter for vLLM).
"""


In [ ]:
# ================= Kaggle T4 fix =================
import os, shutil, glob
STUB_DIR = "/tmp/cuda_stubs"; os.makedirs(STUB_DIR, exist_ok=True)
stub = os.path.join(STUB_DIR, "libcuda.so")
if os.path.exists(stub) or os.path.islink(stub): os.remove(stub)
for c in ["/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib/x86_64-linux-gnu/libcuda.so",
          *glob.glob("/usr/**/libcuda.so*", recursive=True)]:
    if os.path.exists(c) and not os.path.islink(c):
        os.symlink(c, stub); break
os.environ["LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LD_LIBRARY_PATH", "")
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
print("Kaggle T4 fixes applied")


In [ ]:
# ================= INSTALL =================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages",
                "unsloth", "trl<=0.24.0", "datasets<4.4.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--break-system-packages",
                "--no-cache-dir", "peft", "accelerate", "bitsandbytes"], check=True)
import torch
print("PyTorch", torch.__version__, "CUDA", torch.cuda.is_available())


In [ ]:
# ================= CONFIG =================
MODEL_PATH       = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
DATASET_PATH     = "/kaggle/input/logic-based-educational-queries/Logic_Based_Educational_Queries (2).json"
LORA_OUTPUT_DIR  = "/kaggle/working/qwen3_cot_lora_v14"
CHECKPOINT_DIR   = "/kaggle/working/cot_lora_ckpt"
EVAL_OUTPUT_PATH = "/kaggle/working/v14_eval_results.json"

# Split (MUST match v13: CAL_SEED=42, CAL_TRAIN_RATIO=0.80, sample-level)
SEED          = 42
TRAIN_RATIO   = 0.80

MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT   = True

LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

EPOCHS         = 2
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
LEARNING_RATE  = 1e-4
WARMUP_STEPS   = 10
WEIGHT_DECAY   = 0.01
OPTIMIZER      = "adamw_8bit"
LR_SCHEDULER   = "linear"
LOGGING_STEPS  = 10

# Must match inference (we use plain CoT, not thinking-mode tokens)
ENABLE_THINKING_TRAIN = False

# Inline eval
EVAL_MAX_NEW_TOKENS = 512
EVAL_TEMPERATURE    = 0.1   # low temp for evaluation
EVAL_N_SAMPLES      = None  # None = all val; set e.g. 30 for quick sanity

# System prompt -- must be identical at training and inference
SYSTEM_PROMPT = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)
print("Config loaded")


In [ ]:
# ================= LOAD MODEL + ATTACH LoRA =================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED,
)
trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trn:,}/{tot:,} ({100*trn/tot:.2f}%)")


In [ ]:
# ================= BUILD CoT DATASET =================
import json, random, re
from datasets import Dataset

with open(DATASET_PATH, encoding="utf-8") as f:
    raw = json.load(f)
n = len(raw)
print(f"Dataset samples: {n}")

# Sample-level split, deterministic, MATCHES v13.3
rng = random.Random(SEED)
ids = list(range(n)); rng.shuffle(ids)
n_tr = int(n * TRAIN_RATIO)
train_ids = set(ids[:n_tr]); val_ids = set(ids[n_tr:])
train_samples = [raw[i] for i in range(n) if i in train_ids]
val_samples   = [raw[i] for i in range(n) if i in val_ids]
print(f"Train: {len(train_samples)} samples | Val: {len(val_samples)} samples")

def build_user_msg(premises_nl, question):
    lines = ["### Premises"]
    for i, p in enumerate(premises_nl):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"

def build_assistant_msg(explanation, gold):
    # Append "Final Answer: X" unless explanation already ends with such a line
    txt = explanation.strip()
    last = txt.split("\n")[-1].lower()
    if not (re.search(r"\bfinal\s+answer\b", last) or
            re.search(r"\banswer\s*:", last)):
        txt = f"{txt}\n\nFinal Answer: {gold}"
    return txt

# Build records (one per question)
def to_records(samples):
    out = []
    for s in samples:
        nls = s.get("premises-NL", [])
        for q, a, e in zip(s.get("questions", []), s.get("answers", []),
                            s.get("explanation", [])):
            if not e or not str(a).strip():
                continue
            out.append({"conversations": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": build_user_msg(nls, q)},
                {"role": "assistant", "content": build_assistant_msg(e, a)},
            ]})
    return out

train_records = to_records(train_samples)
print(f"Train examples (questions): {len(train_records)}")
print("\n--- Sample training record ---")
print("USER:", train_records[0]["conversations"][1]["content"][:300], "...")
print("\nASSISTANT:", train_records[0]["conversations"][2]["content"][:400])

train_ds = Dataset.from_list(train_records)
print(f"\nDataset: {train_ds}")


In [ ]:
# ================= TRAINER (responses-only) =================
import torch
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def formatting_func(examples):
    convos = examples["conversations"]
    def _fmt(c):
        try:
            return tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False,
                enable_thinking=ENABLE_THINKING_TRAIN)
        except TypeError:
            return tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False)
    if len(convos) > 0 and isinstance(convos[0], dict):
        return [_fmt(convos)]
    return [_fmt(c) for c in convos]

sft = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    optim=OPTIMIZER,
    lr_scheduler_type=LR_SCHEDULER,
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    save_total_limit=1,
    seed=SEED,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)
trainer_obj = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds,
                         args=sft, formatting_func=formatting_func)
trainer_obj = train_on_responses_only(
    trainer_obj,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print(f"Trainer ready | examples={len(train_ds)} | effective batch={BATCH_SIZE*GRAD_ACCUM}")


In [ ]:
# ================= TRAIN + SAVE LoRA =================
import time, os
t0 = time.time()
result = trainer_obj.train()
dur_min = (time.time() - t0) / 60
print(f"Training done in {dur_min:.1f} min | final loss = {result.training_loss:.4f}")

# Save adapter
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"Saved LoRA -> {LORA_OUTPUT_DIR}")
print("Contents:", os.listdir(LORA_OUTPUT_DIR))


In [ ]:
# ================= INLINE EVAL on VAL (vs pure_qwen 33% baseline) =================
import re, json, time
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
print("Model in inference mode")

def extract_final_answer(text):
    if not text: return "UNPARSEABLE"
    # Tier 1: 'Final Answer: X' (case-insensitive)
    m = re.search(r"final\s*answer\s*[:\-]?\s*([A-Da-d]\b|Yes|No|Unknown|YES|NO|UNKNOWN|yes|no|unknown)",
                  text, re.I)
    if m:
        a = m.group(1).strip()
        return a.upper() if a.lower() in ("yes","no","unknown") else a.upper()
    # Tier 2: standalone last line
    for line in reversed(text.strip().split("\n")):
        s = line.strip()
        if s in ("Yes","No","Unknown","A","B","C","D"): return s.upper() if len(s)>1 else s
    # Tier 3: any answer-like token in last 100 chars
    tail = text[-200:]
    for tok in ["Yes","No","Unknown","A","B","C","D"]:
        if re.search(rf"\b{tok}\b", tail):
            return tok.upper() if len(tok)>1 else tok
    return "UNPARSEABLE"

# Build val records (use same code paths as training)
val_records = []
for s in val_samples:
    nls = s.get("premises-NL", [])
    for q_idx, (q, a) in enumerate(zip(s.get("questions", []), s.get("answers", []))):
        val_records.append({"sample_id": s.get("idx", [[]])[0] if False else None,
                            "q_idx": q_idx, "user": build_user_msg(nls, q),
                            "gold": str(a).strip()})

if EVAL_N_SAMPLES is not None:
    val_records = val_records[:EVAL_N_SAMPLES]
print(f"Evaluating {len(val_records)} val questions...")

def gen_one(user_msg):
    msgs = [{"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":user_msg}]
    try:
        ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                            return_tensors="pt", enable_thinking=ENABLE_THINKING_TRAIN).to("cuda")
    except TypeError:
        ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                            return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=EVAL_MAX_NEW_TOKENS,
                         temperature=EVAL_TEMPERATURE, do_sample=(EVAL_TEMPERATURE>0),
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)

t0 = time.time(); results = []
correct = 0
for j, rec in enumerate(val_records):
    raw = gen_one(rec["user"])
    pred = extract_final_answer(raw)
    ok = (str(pred).upper() == str(rec["gold"]).upper())
    if ok: correct += 1
    results.append({"q_idx": rec["q_idx"], "gold": rec["gold"], "pred": pred,
                    "correct": ok, "raw_tail": raw[-300:]})
    if (j+1) % 25 == 0:
        print(f"  [{j+1}/{len(val_records)}] running acc = {correct/(j+1):.1%}  ({(time.time()-t0)/60:.1f} min)")

acc = correct / max(len(val_records), 1)
dur = (time.time() - t0) / 60
print("\n" + "="*70)
print(f"  v14 INLINE EVAL on VAL (matches v13.3 val split)")
print("="*70)
print(f"  Questions evaluated : {len(val_records)}")
print(f"  Correct             : {correct}")
print(f"  ACCURACY            : {acc:.1%}")
print(f"  Duration            : {dur:.1f} min")
print("-"*70)
print(f"  Baseline pure_qwen (v13.3 val) : 33.1%")
print(f"  Improvement                    : {acc - 0.331:+.1%}")
print(f"  Success threshold (>=50%)      : {'PASS' if acc >= 0.50 else 'FAIL'}")
print("="*70)

# Save
summary = {
    "meta": {"version": "v14_cot_lora", "seed": SEED, "train_ratio": TRAIN_RATIO,
             "lora_dir": LORA_OUTPUT_DIR, "duration_min": dur,
             "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")},
    "metrics": {"val_accuracy": acc, "correct": correct, "total": len(val_records),
                "baseline_pure_qwen": 0.331, "improvement_pp": acc - 0.331},
    "per_question": results,
}
with open(EVAL_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"\nSaved eval: {EVAL_OUTPUT_PATH}")

print("\nNEXT STEPS:")
print(f"  - LoRA adapter saved at: {LORA_OUTPUT_DIR}")
print( "  - To integrate in v13.3: in that notebook's config cell set")
print(f"      FORMALIZER_LORA_PATH = '{LORA_OUTPUT_DIR}'  (or wire a new flag)")
print( "    and pass use_lora=True to batch_generate for the CoT step.")
print( "  - Re-run v13.3 -> expect new oracle ceiling + better pure_qwen + better aggregators")
